### Глобальная модель (multi-resist) — `15_model_comparison_global_multi_resist_boost`

Одна табличная модель на **все выбранные `RESIST_IDS`** сразу; наборы признаков **только B+lags и C+lags** (как в `14_model_comparison_multi_calc_resist_warnings.ipynb`).

Лаги по `fact_value` и сдвиги `calc_*` считаются по группе **`(resist_id, operation_id)`** (`group_cols` в `add_calc_shift_lags`). В базовые признаки добавлен **`resist_code`**.

Модели: CatBoost (3), LGBM (2), XGB (2), DT-deep, DT-ccp, RF-default, GBR-default. Тест: июль–декабрь 2025.


In [ ]:
import json
import sys
import time
import copy
from pathlib import Path

_ROOT = Path("..").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from scripts.calc_lag_features import (
    add_calc_shift_lags,
    calc_lag_columns_for_sets,
    DEFAULT_CALC_SHIFTS,
)

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

%matplotlib inline
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning)


#### Конфигурация

In [ ]:
RESIST_ID = 124  # не используется, если задан список RESIST_IDS
RESIST_IDS = [124]  # список resist_id; дубликаты убираются автоматически
GROUP_COLS = ("resist_id", "operation_id")  # группировка рядов для лагов
SUMMARY_TOP = 15  # строк в сводке по качеству

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]

CALC_WHITELIST = [
    "calc_workDayMonth_2",
    "calc_hourInWeek_12",
    "calc_numberWorkHour_1",
    "calc_effective_13",
    "calc_absenteeism_14",
    "calc_employFact_7",
    "calc_resist_10",
]

LAG_FEATURES = [
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
    "rolling_mean_3", "rolling_mean_6", "rolling_mean_12",
]

OUT_XLSX = Path("../data/processed/model_comparison_global_multi_resist_boost.xlsx")


#### Загрузка данных и подготовка (с calc_flat)

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(f"Файлы данных не найдены в {RAW_DIR}")

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record, **calc_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = pd.to_numeric(df_model39[col], errors="coerce")
print(f"calc_* колонки ({len(calc_cols)}):", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

RESIST_IDS = list(dict.fromkeys(RESIST_IDS))
avail = set(int(x) for x in df_model39["resist_id"].dropna().unique())
missing = [r for r in RESIST_IDS if r not in avail]
if missing:
    sample = sorted(avail)[:15]
    raise ValueError(
        f"Нет данных для resist_id={missing}. Примеры доступных id: {sample}..."
    )
print(f"RESIST_IDS ({len(RESIST_IDS)}): {RESIST_IDS}")


#### Метрики

In [ ]:
def calc_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0, 100,
        )
    tochnost = round(float(np.mean(acc)), 2)
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = round(max(0.0, (1 - cv)) * 100, 2)
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = round(2 * tochnost * stab / (tochnost + stab), 2)
    return tochnost, stab, kachestvo


#### Обучение: одна модель на все строки (B+lags / C+lags)

In [ ]:
def add_fact_lag_columns(
    df: pd.DataFrame,
    lag_feature_names: list[str],
    group_cols: tuple[str, ...],
) -> pd.DataFrame:
    """Лаги по fact_value по группе group_cols (например resist_id + operation_id)."""
    out = df.copy()
    g = out.groupby(list(group_cols), sort=False)
    for name in lag_feature_names:
        if name.startswith("lag_"):
            k = int(name.split("_", 1)[1])
            out[name] = g["fact_value"].shift(k)
        elif name.startswith("rolling_mean_"):
            w = int(name.replace("rolling_mean_", ""))
            out[name] = g["fact_value"].transform(
                lambda s, ww=w: s.shift(1).rolling(window=ww, min_periods=1).mean()
            )
        else:
            raise ValueError(f"Неизвестный признак лага: {name}")
    num_cols = out.select_dtypes(include=["number"]).columns
    out[num_cols] = out[num_cols].fillna(0)
    out = out.infer_objects(copy=False)
    return out


def _build_model_configs(cat_idx: list[int]) -> dict:
    """Только выбранные конфигурации; CatBoost с cat_features."""
    return {
        "CB-default": CatBoostRegressor(
            iterations=500, depth=6, learning_rate=0.1,
            verbose=0, random_state=42, cat_features=cat_idx,
        ),
        "CB-deep": CatBoostRegressor(
            iterations=800, depth=8, learning_rate=0.05,
            verbose=0, random_state=42, cat_features=cat_idx,
        ),
        "CB-conservative": CatBoostRegressor(
            iterations=300, depth=4, learning_rate=0.1,
            l2_leaf_reg=5, verbose=0, random_state=42, cat_features=cat_idx,
        ),
        "LGBM-default": LGBMRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            verbose=-1, random_state=42,
        ),
        "LGBM-tuned": LGBMRegressor(
            n_estimators=800, max_depth=8, learning_rate=0.05,
            num_leaves=63, min_child_samples=10, verbose=-1, random_state=42,
        ),
        "XGB-default": XGBRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            verbosity=0, random_state=42,
        ),
        "XGB-regularized": XGBRegressor(
            n_estimators=800, max_depth=8, learning_rate=0.05,
            reg_alpha=0.1, reg_lambda=5, verbosity=0, random_state=42,
        ),
        "DT-deep": DecisionTreeRegressor(max_depth=12, min_samples_leaf=2, random_state=42),
        "DT-ccp": DecisionTreeRegressor(
            max_depth=15, min_samples_leaf=2, ccp_alpha=0.02, random_state=42,
        ),
        "RF-default": RandomForestRegressor(
            n_estimators=300, max_depth=12, min_samples_leaf=2,
            random_state=42, n_jobs=-1,
        ),
        "GBR-default": GradientBoostingRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            random_state=42,
        ),
    }


df = df_model39[df_model39["resist_id"].isin(RESIST_IDS)].copy()
df = df.sort_values(list(GROUP_COLS) + ["year", "month"], kind="mergesort").reset_index(drop=True)
df["op_code"] = df["operation_id"].astype("category").cat.codes
df["resist_code"] = df["resist_id"].astype("category").cat.codes

df = add_fact_lag_columns(df, LAG_FEATURES, GROUP_COLS)

miss = df.isnull().sum()
miss = miss[miss > 0]
if len(miss) > 0:
    print("Пропуски (до calc_*):", miss.to_dict())
    df = df.fillna(0)

df, _lag_calc_cols, calc_keys_used = add_calc_shift_lags(
    df, drop_scalar_calc=True, group_cols=GROUP_COLS,
)
b_flat, c_flat = calc_lag_columns_for_sets(calc_keys_used, whitelist=CALC_WHITELIST)
print(
    f"flatten calc_* (lag-сдвиги): ключей {len(calc_keys_used)}; "
    f"набор B: {len(b_flat)}, набор C: {len(c_flat)}; shift-месяцы: {DEFAULT_CALC_SHIFTS}"
)

BASE_CORE = ["resist_code", "op_code", "month", "year", "workDayMonth_2"]
BASE_WITH_MINUTE = BASE_CORE + ["calculate_minute"]
FEATURE_SETS = {
    "B+lags": BASE_WITH_MINUTE + b_flat + LAG_FEATURES,
    "C+lags": BASE_WITH_MINUTE + c_flat + LAG_FEATURES,
}

for fs_name, fs_cols in FEATURE_SETS.items():
    miss = [c for c in fs_cols if c not in df.columns]
    if miss:
        raise ValueError(f"Не хватает колонок для {fs_name}: {miss[:20]}...")
    print(f"  {fs_name}: {len(fs_cols)} признаков")

mask_test = (df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS))
mask_train = ~mask_test
y_train = df.loc[mask_train, "fact_value"]
y_test = df.loc[mask_test, "fact_value"]
n_test = int(mask_test.sum())
n_train = int(mask_train.sum())
print(f"Train: {n_train} строк, Test: {n_test} строк")
t_fp, s_fp, q_fp = calc_metrics(y_test.values, df.loc[mask_test, "fact_predict"].values)
print(f"Baseline fact_predict: Точность={t_fp}%, Стабильность={s_fp}%, Качество={q_fp}%")
results: list = []
preds_store: dict[tuple[str, str], np.ndarray] = {}

if n_train == 0:
    raise RuntimeError("Train пуст — нельзя обучить модели.")

for fs_name, fs_cols in FEATURE_SETS.items():
    cat_idx = [fs_cols.index("op_code"), fs_cols.index("resist_code")]
    model_configs = _build_model_configs(cat_idx)
    X_train_fs = df.loc[mask_train, fs_cols]
    X_test_fs = df.loc[mask_test, fs_cols]

    for config_name, model in model_configs.items():
        combo_key = (config_name, fs_name)
        t0 = time.perf_counter()
        m = copy.deepcopy(model)
        m.fit(X_train_fs, y_train)
        y_pred = np.maximum(m.predict(X_test_fs), 0)
        elapsed = round(time.perf_counter() - t0, 2)
        preds_store[combo_key] = y_pred
        tochnost, stab, kachestvo = calc_metrics(y_test.values, y_pred)
        results.append(
            {
                "Конфигурация": config_name,
                "Набор признаков": fs_name,
                "Тип": "табличная (global)",
                "Точность, %": tochnost,
                "Стабильность, %": stab,
                "Качество, %": kachestvo,
                "Время, сек": elapsed,
                "Точн. fact_predict, %": t_fp,
                "Стаб. fact_predict, %": s_fp,
                "Кач. fact_predict, %": q_fp,
            }
        )
        print(f"{config_name} | {fs_name}: Качество={kachestvo}% ({elapsed}s)")

df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Качество, %", ascending=False).reset_index(drop=True)
print("\n--- Топ по Качеству ---")
print(df_results.head(SUMMARY_TOP).to_string(index=False))

# Разбивка по resist_id (y_pred выровнен только по строкам теста)
df_test = df.loc[mask_test]
per_rid_rows: list = []
for rid in RESIST_IDS:
    sub = df_test["resist_id"].values == rid
    if not sub.any():
        continue
    yt = df_test["fact_value"].values[sub]
    for (config_name, fs_name), y_pred in preds_store.items():
        yp = y_pred[sub]
        t, s, k = calc_metrics(yt, yp)
        per_rid_rows.append(
            {
                "resist_id": rid,
                "Конфигурация": config_name,
                "Набор признаков": fs_name,
                "Точность, %": t,
                "Стабильность, %": s,
                "Качество, %": k,
            }
        )
df_per_rid = pd.DataFrame(per_rid_rows)

OUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    df_results.to_excel(w, sheet_name="Итог", index=False)
    df_per_rid.to_excel(w, sheet_name="По resist_id", index=False)
print(f"\nСохранено: {OUT_XLSX.resolve()}")
